<a href="https://colab.research.google.com/github/Likhitha-26-AI/MedRoute/blob/main/MedRoute_MedGemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  MedRoute — Agentic Patient Triage & Care Navigation System

**Built with Fine-tuned MedGemma · Google Health AI Developer Foundations (HAI-DEF)**

---

##  Overview

MedRoute is a 4-agent AI pipeline powered by fine-tuned MedGemma-4B that takes a patient from first symptom description to a clinician-ready care plan in under 2 minutes.

**The 4 Agents:**
- **Agent 1 — Clinical Intake:** Extracts structured clinical information and asks a targeted follow-up question
- **Agent 2 — Triage Classifier:** Fine-tuned MedGemma classifies urgency (0=Non-Urgent, 1=Less Urgent, 2=Urgent, 3=Emergency)
- **Agent 3 — Care Navigator:** Generates personalized care plan with next steps and red flags
- **Agent 4 — Clinical Handoff:** Produces SOAP-format physician note

---

## ⚠️ IMPORTANT — Read Before Running

### GPU is REQUIRED
MedGemma-4B is a 4 billion parameter model. **It cannot run on CPU.** Running on CPU will cause the notebook to crash or take hours to respond.

**Before running any cell:**
1. Click **Runtime** in the top menu
2. Click **Change runtime type**
3. Under **Hardware accelerator** select **T4 GPU**
4. Click **Save**
5. Now you are ready to run the cells

### HuggingFace Token is REQUIRED
MedGemma is a gated model — you need a HuggingFace account and token to access it.

**Steps to get your token:**
1. Go to [huggingface.co](https://huggingface.co) and create a free account
2. Go to [huggingface.co/google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it) and request access
3. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
4. Click **New token**
5. Give it a name, set permission to **Read** (Read is enough — you only need Write if uploading models)
6. Copy the token

**How to store your token safely in Colab (DO NOT paste it directly in code):**
1. Look at the **left sidebar** in Colab
2. Click the **🔑 key icon** (Secrets)
3. Click **Add new secret**
4. Name: `HF_TOKEN`
5. Value: paste your HuggingFace token here
6. Toggle **Notebook access** ON
7. Click **Done**

Your token is now stored safely and the code will read it automatically.

---

## 🗂️ Table of Contents
1. Install Requirements
2. Load HuggingFace Token
3. Load Base MedGemma Model
4. Load Fine-tuned MedRoute Adapter
5. Run the 4-Agent Demo (Gradio UI)
6. Performance Evaluation (Optional)
7. Fine-tuning from Scratch (Optional — for advanced users)

---

## Step 1 — Install Requirements

This cell installs all required libraries. Run this first and wait for it to complete before running any other cell.

**Expected time:** 2-3 minutes

In [ ]:
# Install all required libraries
!pip install -q transformers==4.47.0
!pip install -q accelerate
!pip install -q peft --upgrade
!pip install -q bitsandbytes
!pip install -q trl
!pip install -q datasets
!pip install -q gradio
!pip install -q huggingface_hub

print("✅ All requirements installed successfully!")

## Step 2 — Load HuggingFace Token

This cell reads your HuggingFace token from Colab Secrets.

**Make sure you have added your token to Colab Secrets before running this cell.**
- Key icon on the left sidebar → Add secret → Name: `HF_TOKEN` → Paste your token → Toggle ON

Use a **Read** token — Write permission is only needed if you want to upload your own models to HuggingFace.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Read token safely from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets. Please add it using the key icon on the left sidebar.")

# Login to HuggingFace
login(HF_TOKEN)
print("✅ Successfully logged in to HuggingFace!")

## Step 3 — Load Base MedGemma Model

This cell loads the base MedGemma-4B model using 4-bit quantization to fit within Colab's free GPU memory limits.

**What is 4-bit quantization?**
It compresses the model from ~8GB to ~4GB so it fits on a free T4 GPU (which has 15GB VRAM). There is a very small quality tradeoff but it is negligible for this use case.

**Expected time:** 5-10 minutes depending on internet speed

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Verify GPU is available
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Please go to Runtime → Change runtime type → Select T4 GPU and restart.")

print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
print(f"   Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_id = "google/medgemma-4b-it"

# Load tokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer loaded!")

# 4-bit quantization config to fit on free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load base model
print("\nLoading MedGemma-4B base model (this takes 5-10 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
print("✅ Base MedGemma model loaded successfully!")

## Step 4 — Load Fine-tuned MedRoute Adapter

This cell loads the fine-tuned LoRA adapter on top of the base MedGemma model.

**What is a LoRA adapter?**
Instead of saving a full copy of the fine-tuned model (which would be 8GB), LoRA saves only the changes made during fine tuning (13MB). The adapter is applied on top of the base model at inference time.

The fine-tuned adapter is publicly available at: huggingface.co/spark-2026/medroute-medgemma-finetuned

**Expected time:** 1-2 minutes

In [ ]:
from peft import PeftModel

print("Loading fine-tuned MedRoute LoRA adapter...")
ft_model = PeftModel.from_pretrained(
    model,
    "spark-2026/medroute-medgemma-finetuned",
    offload_folder="/tmp/offload"
)
ft_model.eval()
print("✅ Fine-tuned MedRoute model loaded successfully!")
print("\nMedRoute is ready to use!")

## Step 5 — Run the 4-Agent Demo (Gradio UI)

This cell launches the full MedRoute 4-agent pipeline with a professional Gradio interface.

**How to use:**
1. Run this cell
2. Wait for a public link ending in `gradio.live` to appear
3. Click the link to open the MedRoute UI
4. Type your symptoms in the first box
5. Optionally add a follow-up answer in the second box
6. Click **Run Full Triage Assessment**
7. Wait 1-2 minutes for all 4 agents to respond

**Test these example symptoms:**
- Emergency: `I have severe chest pain radiating to my left arm, started 20 minutes ago. I feel dizzy and short of breath.`
- Urgent: `I have a high fever of 39.5 degrees, severe headache and neck stiffness since this morning.`
- Non-Urgent: `I have a mild sore throat and slight runny nose for the past 2 days. No fever.`

**Note:** The public gradio.live link is valid for 72 hours. After that rerun this cell to get a new link.

In [ ]:
import gradio as gr
import torch

def run_agent1_collect(symptoms):
    prompt = f"""<start_of_turn>user
You are a clinical intake nurse. A patient describes their symptoms. Extract key medical information and ask ONE important follow-up question.
Patient says: {symptoms}
Respond in this format:
UNDERSTOOD: [summarize symptoms in clinical terms]
FOLLOW-UP: [one important question]<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def run_agent2_triage(symptoms, followup):
    prompt = f"""<start_of_turn>user
You are a medical triage assistant. Reply with ONLY a single digit: 0 (Non-urgent), 1 (Less urgent), 2 (Urgent), or 3 (Emergency).
Patient complaint: {symptoms}
Additional info: {followup}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    result = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    for char in result:
        if char in "0123":
            return char
    return "1"

def run_agent3_navigate(symptoms, triage_level):
    level_names = {"0": "Non-Urgent", "1": "Less Urgent", "2": "Urgent", "3": "Emergency"}
    prompt = f"""<start_of_turn>user
You are a clinical care navigator. Triage level: {level_names.get(triage_level, "Urgent")}.
Patient symptoms: {symptoms}
Provide:
1. WHERE TO GO: Specific care setting
2. NEXT STEPS: 3 immediate actions
3. RED FLAGS: 2 warning signs requiring emergency care
4. PREPARE: What to tell the doctor<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def run_agent4_summary(symptoms, followup, triage_level):
    level_names = {"0": "Non-Urgent", "1": "Less Urgent", "2": "Urgent", "3": "Emergency"}
    prompt = f"""<start_of_turn>user
Generate a concise clinical SOAP note for a physician. No placeholders.
Chief Complaint: {symptoms}
Patient Added: {followup}
Triage Level: {level_names.get(triage_level, "Urgent")}

Write exactly 4 short sections, keep each to 2 sentences maximum:
S (Subjective): Chief complaint in patient's own words
O (Objective): Key clinical observations based on reported symptoms
A (Assessment): Most likely diagnosis or clinical impression
P (Plan): Immediate next steps and treatment plan<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def run_full_pipeline(symptoms, followup_answer):
    if not symptoms.strip():
        return "", "", "", ""
    agent1_out = run_agent1_collect(symptoms)
    triage_level = run_agent2_triage(symptoms, followup_answer)
    care_plan = run_agent3_navigate(symptoms, triage_level)
    summary = run_agent4_summary(symptoms, followup_answer, triage_level)
    triage_display = {
        "0": "🟢  NON-URGENT — Schedule routine appointment. No immediate danger.",
        "1": "🟡  LESS URGENT — See a doctor within 24 hours. Monitor symptoms closely.",
        "2": "🟠  URGENT — Visit urgent care within 2 hours. Bring someone with you.",
        "3": "🔴  EMERGENCY — Call 911 immediately. Do not drive yourself."
    }.get(triage_level, "🟡  LESS URGENT — See a doctor soon.")
    return agent1_out, triage_display, care_plan, summary

theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("DM Mono"), "monospace"],
).set(
    body_background_fill="#F8F9FA",
    block_background_fill="#FFFFFF",
    block_border_width="1px",
    block_border_color="#E2E8F0",
    block_radius="12px",
    block_shadow="0 1px 4px rgba(0,0,0,0.06)",
    button_primary_background_fill="#1D4ED8",
    button_primary_background_fill_hover="#1E40AF",
    button_primary_text_color="#FFFFFF",
    button_primary_border_color="#1D4ED8",
    input_background_fill="#F8FAFC",
    input_border_color="#CBD5E0",
    input_border_width="1px",
    input_radius="8px",
)

with gr.Blocks(theme=theme, title="MedRoute — Clinical Triage System") as demo:
    gr.Markdown("""
    # 🏥 MedRoute — Clinical Triage & Care Navigation
    **AI-powered patient assessment using fine-tuned MedGemma · Google HAI-DEF · 4-Agent Pipeline**
    ---
    """)
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Patient Intake")
            symptoms = gr.Textbox(
                label="Describe your symptoms",
                placeholder="e.g. I have severe chest pain radiating to my left arm, started 20 minutes ago...",
                lines=4
            )
            followup = gr.Textbox(
                label="Answer follow-up question (after Agent 1 responds)",
                placeholder="Your response...",
                lines=2
            )
            btn = gr.Button("Run Full Triage Assessment", variant="primary", size="lg")
    gr.Markdown("---")
    gr.Markdown("### Agent Outputs")
    with gr.Row():
        with gr.Column():
            agent1_out = gr.Textbox(label="Agent 1 — Symptom Assessment & Follow-up", lines=4, interactive=False)
    with gr.Row():
        with gr.Column():
            triage_out = gr.Textbox(label="Agent 2 — Triage Classification (Fine-tuned MedGemma)", lines=2, interactive=False)
    with gr.Row():
        with gr.Column():
            care_out = gr.Textbox(label="Agent 3 — Care Navigation Plan", lines=8, interactive=False)
    with gr.Row():
        with gr.Column():
            summary_out = gr.Textbox(label="Agent 4 — Clinical Handoff Summary (SOAP Note)", lines=6, interactive=False)
    gr.Markdown("""
    ---
    *⚠️ MedRoute is a demonstration system. Always consult a qualified healthcare professional for medical advice.*
    """)
    btn.click(
        fn=run_full_pipeline,
        inputs=[symptoms, followup],
        outputs=[agent1_out, triage_out, care_out, summary_out]
    )

demo.launch(share=True)

## Step 6 — Performance Evaluation (Optional)

This cell evaluates the fine-tuned model on 10 representative test cases spanning all 4 triage levels.

**Expected result:** 9/10 (90%) accuracy

Run this cell after Step 4 to reproduce our validation results.

In [ ]:
# Performance Evaluation on 10 representative test cases
test_cases = [
    {"age": 79, "hr": 147, "bp": 158, "o2": 96, "temp": 39.35, "pain": 10, "chronic": 4, "er_visits": 2, "arrival": "ambulance", "expected": 3},
    {"age": 17, "hr": 95, "bp": 147, "o2": 97, "temp": 36.48, "pain": 1, "chronic": 0, "er_visits": 0, "arrival": "walk-in", "expected": 0},
    {"age": 56, "hr": 84, "bp": 147, "o2": 92, "temp": 37.55, "pain": 4, "chronic": 4, "er_visits": 4, "arrival": "walk-in", "expected": 1},
    {"age": 39, "hr": 58, "bp": 107, "o2": 99, "temp": 36.26, "pain": 2, "chronic": 1, "er_visits": 1, "arrival": "walk-in", "expected": 0},
    {"age": 51, "hr": 87, "bp": 128, "o2": 98, "temp": 37.74, "pain": 5, "chronic": 2, "er_visits": 2, "arrival": "walk-in", "expected": 1},
    {"age": 65, "hr": 130, "bp": 170, "o2": 88, "temp": 38.5, "pain": 8, "chronic": 3, "er_visits": 1, "arrival": "ambulance", "expected": 3},
    {"age": 45, "hr": 100, "bp": 180, "o2": 95, "temp": 37.2, "pain": 7, "chronic": 2, "er_visits": 0, "arrival": "walk-in", "expected": 2},
    {"age": 28, "hr": 105, "bp": 125, "o2": 96, "temp": 39.5, "pain": 6, "chronic": 0, "er_visits": 0, "arrival": "walk-in", "expected": 2},
    {"age": 72, "hr": 120, "bp": 160, "o2": 94, "temp": 38.0, "pain": 7, "chronic": 4, "er_visits": 3, "arrival": "ambulance", "expected": 3},
    {"age": 22, "hr": 72, "bp": 115, "o2": 99, "temp": 36.5, "pain": 1, "chronic": 0, "er_visits": 0, "arrival": "walk-in", "expected": 0},
]

level_names = {0: "Non-Urgent", 1: "Less Urgent", 2: "Urgent", 3: "Emergency"}
correct = 0
print("=" * 80)
print(f"{'Case':<6} {'Expected':<20} {'Predicted':<20} {'Result':<10}")
print("=" * 80)

for i, case in enumerate(test_cases):
    prompt = f"""<start_of_turn>user
You are a medical triage assistant. Reply with ONLY a single digit: 0 (Non-urgent), 1 (Less urgent), 2 (Urgent), or 3 (Emergency).
Patient: Age {case['age']}, HR {case['hr']}, BP {case['bp']}, O2 {case['o2']}%, Temp {case['temp']}C, Pain {case['pain']}/10, Chronic conditions: {case['chronic']}, ER visits: {case['er_visits']}, Arrival: {case['arrival']}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    result = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    predicted = int(next((c for c in result if c in "0123"), "1"))
    is_correct = predicted == case["expected"]
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    print(f"{i+1:<6} {level_names[case['expected']]:<20} {level_names[predicted]:<20} {status}")

print("=" * 80)
print(f"\nFinal Accuracy: {correct}/10 ({correct*10}%)")

## Step 7 — Fine-tuning from Scratch (Optional — Advanced Users)

This section shows how the MedRoute model was fine-tuned from scratch.

**⚠️ Warning:** Fine-tuning takes approximately 10 hours on a T4 GPU. Make sure your Colab session won't disconnect.

**You do NOT need to run this section to use MedRoute.** The fine-tuned adapter is already available at huggingface.co/spark-2026/medroute-medgemma-finetuned

**Dataset:** Synthetic Medical Triage Priority Dataset — 18,000 patient records with features including age, heart rate, blood pressure, oxygen saturation, temperature, pain level, chronic condition count, ER visit history, and arrival mode.

**LoRA Configuration:**
- Rank (r): 8
- Alpha: 16
- Dropout: 0.05
- Target modules: q_proj, v_proj
- Training steps: 2,025
- Learning rate: 2e-4
- Precision: bf16
- Final training loss: 0.2792 (83% reduction from 1.6509)

In [ ]:
# ============================================================
# OPTIONAL — Fine-tuning from Scratch
# Only run this if you want to retrain the model yourself
# Expected time: ~10 hours on T4 GPU
# ============================================================

from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

# Load dataset
print("Loading dataset...")
dataset = load_dataset("emirhanakku/synthetic-medical-triage-priority-dataset")
print(f"✅ Dataset loaded: {len(dataset['train'])} training records")

# Format dataset for fine-tuning
def format_example(example):
    prompt = f"""<start_of_turn>user
You are a medical triage assistant. Reply with ONLY a single digit: 0 (Non-urgent), 1 (Less urgent), 2 (Urgent), or 3 (Emergency).
Patient: Age {example['age']}, HR {example['heart_rate']}, BP {example['blood_pressure']}, O2 {example['oxygen_saturation']}%, Temp {example['temperature']}C, Pain {example['pain_level']}/10, Chronic conditions: {example['chronic_conditions']}, ER visits: {example['previous_er_visits']}, Arrival: {example['arrival_mode']}<end_of_turn>
<start_of_turn>model
{example['triage_priority']}<end_of_turn>"""
    return {"text": prompt}

formatted_dataset = dataset["train"].map(format_example)
print("✅ Dataset formatted!")

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="/tmp/medroute-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_ratio=0.03,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none"
)

# Trainer
trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer
)

print("Starting fine-tuning... This will take ~10 hours on T4 GPU")
trainer.train()

# Save model
trainer.save_model("/tmp/medroute-finetuned")
tokenizer.save_pretrained("/tmp/medroute-finetuned")
print("✅ Fine-tuning complete! Model saved to /tmp/medroute-finetuned")

---

##  Project Details

| Item | Details |
|---|---|
| Base Model | google/medgemma-4b-it |
| Fine-tuned Model | huggingface.co/spark-2026/medroute-medgemma-finetuned |
| Live Demo | huggingface.co/spaces/spark-2026/MedRoute |
| Dataset | Synthetic Medical Triage Priority Dataset (18,000 records) |
| Method | LoRA (r=8, alpha=16) |
| Training Loss | 1.6509 → 0.2792 (83% reduction) |
| Test Accuracy | 90% (9/10 cases) |
| Competition | MedGemma Impact Challenge — Google HAI-DEF |

## ⚠️ Disclaimer

MedRoute is a research demonstration system built for the MedGemma Impact Challenge. It is not intended for clinical use. Always consult a qualified healthcare professional for medical advice.

---

*Built by G. Likhitha Rao — MedGemma Impact Challenge 2026*